# 不使用@tool的方式定义工具

举例：

In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from config.config import DeepSeekModel

model = init_chat_model(
    base_url=DeepSeekModel.deepseek_base_url,
    api_key=DeepSeekModel.deepseek_api_key,
    model=DeepSeekModel.deepseek_model,
    model_provider="openai"
)

def get_weather(city: str) -> str:
    return f"{city}天气晴朗"

# 将函数绑定到模型
model_with_tools = model.bind_tools([get_weather])

# 调用
response = model_with_tools.invoke("沈阳天气怎么样？")

rprint(response)

AIMessage(
    content='我来帮您查询沈阳的天气情况。',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 66,
            'prompt_tokens': 286,
            'total_tokens': 352,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 19,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 286
        },
        'model_provider': 'openai',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': '7ef7dd9d-a4fe-47f9-b0a6-441f3e799919',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a117f1-c2d8-7ae0-a1bd-0895ac74a517-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '沈阳'},
            'id': 'call_00_k0DAzeP7KMh3A4WegMbk7948',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 286,
        'output_tokens': 66,
        'total_tokens': 352,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 19}
    }
)

## 2 工具各部分详解

## 2.1 了解convert_to_openai_tool

model.bind_tools([get_weather])底层会调用convert_to_openai_tool生成工具描述，所以我们可以直接调用后者查看解析后的工具描述


In [7]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city: str) -> str:
    """
    调用天气api查看指定城市天气

    Args:
        city:城市名

    Returns:
        返回指定城市天气
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '调用天气api查看指定城市天气',
        'parameters': {
            'properties': {'city': {'description': '城市名', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}